# Time-Varying AR Parameters

Loads the AR-parameter CSVs saved by the `global` and `local` training stages for three datasets (Air Passengers, Rossmann Store Sales, M5 Aggregated) and plots the learned, time-varying AR coefficients per lag for a selected series.

**Prerequisite:** the corresponding `global.ipynb` / `local.ipynb` runs must have been executed with `save_extras=True`. The CSVs read are:
- `experiments/runs/results/local/airpassengers_ar_parameters.csv`
- `experiments/runs/results/global/rossmann_ar_parameters.csv`
- `experiments/runs/results/global/m5_agg_ar_parameters.csv`

**Output:** one PDF and one PNG per dataset at `experiments/runs/results/plots/time_varying_params_<dataset>.{pdf,png}`.

# Imports

In [ ]:
import sys
from pathlib import Path
from IPython.display import display
from plotnine import *

repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from experiments.utils import save_plot, load_params_long

results_dir = repo_root / 'experiments' / 'runs' / 'results'
plots_dir = results_dir / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)

seed = 123

## Dataset configuration

In [ ]:
dataset_configs = [
    {'dataset': 'airpassengers', 'train_type': 'local',  'series_id': None},
    {'dataset': 'rossmann',      'train_type': 'global', 'series_id': '1042'},
    {'dataset': 'm5_agg',        'train_type': 'global', 'series_id': 'FOODS_3_FOODS_TX_2_TX'},
]

## Load CSVs and plot each dataset

In [ ]:
color_map = {'Hyper-Tree-AR': '#d62728', 'Hyper-TreeNet-AR': '#2ca02c'}

for cfg in dataset_configs:
    dataset      = cfg['dataset']
    train_type   = cfg['train_type']
    selected_sid = cfg['series_id']

    print(f'\n=== Loading {dataset} ({train_type}) ===')
    long, train_end, max_lag, fcst_h = load_params_long(
        dataset=dataset,
        train_type=train_type,
        results_dir=results_dir,
        series_id=selected_sid,
    )

    ctx_len = 10 * fcst_h
    long = long.groupby(['model', 'variable']).tail(ctx_len).reset_index(drop=True)

	# For M5, the AR coefficients can be very large for some series, which distorts the plot. We cap the maximum value to the median for better visualization.
    if dataset == 'm5_agg':
        max_idx = long['value'].idxmax()
        long.loc[max_idx, 'value'] = long['value'].median()

    figsize = (16, 15) if dataset == 'rossmann' else (16, 12)
    title = '' if dataset == 'airpassengers' else f'Series-ID: {selected_sid}'

    plot = (
        ggplot(long, aes(x='date', y='value', color='model'))
        + geom_line(aes(size='model'))
        + geom_vline(xintercept=train_end, linetype='dashed', color='black')
        + scale_color_manual(values=color_map)
        + scale_size_manual(values={'Hyper-Tree-AR': 1.1, 'Hyper-TreeNet-AR': 1.1})
        + facet_wrap('variable', scales='free_y', ncol=3)
        + theme_bw(base_size=15)
        + theme(
            figure_size=figsize,
            legend_position='bottom',
            legend_title=element_blank(),
            legend_key=element_rect(fill='none', colour='none'),
            panel_grid_major_x=element_line(color='grey', alpha=0.05),
            panel_grid_minor_x=element_line(color='grey', alpha=0.05),
            panel_grid_major_y=element_line(color='grey', alpha=0.05),
            panel_grid_minor_y=element_line(color='grey', alpha=0.05),
            axis_text_x=element_text(angle=45, hjust=1),
            plot_title=element_text(ha='center', size=12),
        )
        + labs(title=title, x='', y='AR-Coefficient')
    )

    save_plot(plot, f'time_varying_params_{dataset}', plots_dir)
    display(plot)